# Notebook 01 — Data Ingestion
**Bluestock Fintech Mutual Fund Analytics Capstone**

In [1]:

import pandas as pd
import requests, json
from pathlib import Path

ROOT = Path().resolve().parent
RAW  = ROOT / 'data' / 'raw'

datasets = {
    'fund_master'          : '01_fund_master.csv',
    'nav_history'          : '02_nav_history.csv',
    'aum_by_fund_house'    : '03_aum_by_fund_house.csv',
    'monthly_sip_inflows'  : '04_monthly_sip_inflows.csv',
    'category_inflows'     : '05_category_inflows.csv',
    'industry_folio_count' : '06_industry_folio_count.csv',
    'scheme_performance'   : '07_scheme_performance.csv',
    'investor_transactions': '08_investor_transactions.csv',
    'portfolio_holdings'   : '09_portfolio_holdings.csv',
    'benchmark_indices'    : '10_benchmark_indices.csv',
}

dfs = {}
for name, fname in datasets.items():
    df = pd.read_csv(RAW / fname)
    dfs[name] = df
    print(f"\n{'='*55}")
    print(f"  {name.upper()}")
    print(f"  Shape: {df.shape}")
    print(f"  Dtypes:\n{df.dtypes.to_string()}")
    print(f"  Head:\n{df.head(2).to_string()}")



  FUND_MASTER
  Shape: (40, 15)
  Dtypes:
amfi_code               int64
fund_house                str
scheme_name               str
category                  str
sub_category              str
plan                      str
launch_date               str
benchmark                 str
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager              str
risk_category             str
sebi_category_code        str
  Head:
   amfi_code       fund_house                                scheme_name category sub_category     plan launch_date      benchmark  expense_ratio_pct  exit_load_pct  min_sip_amount  min_lumpsum_amount   fund_manager risk_category sebi_category_code
0     119551  SBI Mutual Fund  SBI Bluechip Fund - Regular Plan - Growth   Equity    Large Cap  Regular  2006-02-14  NIFTY 100 TRI               1.54            1.0             500                1000  Sohini Andani      Moderate               EC01
1

## Live NAV Fetch from mfapi.in

In [2]:

SCHEMES = {
    125497: 'HDFC Top 100 Direct',
    119551: 'SBI Bluechip',
    120503: 'ICICI Bluechip',
    118632: 'Nippon Large Cap',
    119092: 'Axis Bluechip',
    120841: 'Kotak Bluechip',
}

for code_id, name in SCHEMES.items():
    try:
        r = requests.get(f'https://api.mfapi.in/mf/{code_id}', timeout=10)
        data = r.json()
        nav_records = data.get('data', [])
        df_live = pd.DataFrame(nav_records)
        df_live.to_csv(RAW / f'nav_live_{code_id}.csv', index=False)
        print(f'{name} ({code_id}): {len(df_live)} records fetched')
    except Exception as e:
        print(f'{name} ({code_id}): fetch failed - {e}')


HDFC Top 100 Direct (125497): fetch failed - Expecting value: line 1 column 1 (char 0)
SBI Bluechip (119551): fetch failed - Expecting value: line 1 column 1 (char 0)
ICICI Bluechip (120503): fetch failed - Expecting value: line 1 column 1 (char 0)


Nippon Large Cap (118632): fetch failed - Expecting value: line 1 column 1 (char 0)
Axis Bluechip (119092): fetch failed - Expecting value: line 1 column 1 (char 0)
Kotak Bluechip (120841): fetch failed - Expecting value: line 1 column 1 (char 0)


## Data Quality Summary

In [3]:

print("DATA QUALITY SUMMARY")
print("="*50)
for name, df in dfs.items():
    nulls = df.isnull().sum().sum()
    dups  = df.duplicated().sum()
    print(f'{name:<30} rows={len(df):<6} nulls={nulls:<5} dups={dups}')


DATA QUALITY SUMMARY
fund_master                    rows=40     nulls=0     dups=0
nav_history                    rows=46000  nulls=0     dups=0
aum_by_fund_house              rows=90     nulls=0     dups=0
monthly_sip_inflows            rows=48     nulls=12    dups=0
category_inflows               rows=144    nulls=0     dups=0
industry_folio_count           rows=21     nulls=0     dups=0
scheme_performance             rows=40     nulls=0     dups=0
investor_transactions          rows=32778  nulls=0     dups=0
portfolio_holdings             rows=322    nulls=0     dups=0
benchmark_indices              rows=8050   nulls=0     dups=0
